# bsc_00 — Bootstrap & Gate 0

**Phase 0.** Chuan bi va **kiem chung metric** truoc khi do bat cu thu gi.

Mục tiêu Gate 0: `bsc.metrics` **tái tạo bảng đã công bố** (femoral_cart Dice 0.891,
ASSD 0.21) trong ±0.005. Nếu không tái tạo được ⇒ **DỪNG**: không thể đo hiệu ứng
0.1mm bằng metric chưa kiểm chứng.

Việc trong notebook này:
1. Mount Drive, clone repo, set `BSC_ROOT` (Drive-first).
2. Giải nén B0/B1/B2 từ 15 GB zip → upload lên `BSC_ROOT/baselines/`.
3. Kiểm `splits_zib_v1.json` đã ghim.
4. **Gate 0**: tính lại metric trên prediction B0 đã có, đối chiếu bảng đã công bố.
5. Điều tra KL grade từ HuggingFace `info.zip`.


In [2]:
# ============================================================
# CELL CONFIG CHUAN - tai dung o MOI notebook bsc_*
# Drive-first: MOI artifact nam duoi BSC_ROOT. KHONG ghi vao /content/.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/AIVIETNAM-AIO-Tuan/bsCart-net.git"   # <-- doi thanh repo cua ban
REPO_DIR = "/content/repo"

import os, sys
if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
sys.path.insert(0, REPO_DIR)

# Thu can cho Colab (may local da co scipy/skimage/numpy)
!pip install -q nibabel SimpleITK 2>/dev/null

BSC_ROOT = "/content/drive/MyDrive/bsc"          # goc artifact - TAT CA nam duoi day
os.makedirs(BSC_ROOT, exist_ok=True)
for sub in ["splits","baselines","geom","raydb","atlas","runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)

# Duong du lieu cu (READ-ONLY - khong bao gio ghi de)
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"   # <-- kiem lai duong nay
print("BSC_ROOT =", BSC_ROOT)
print("Cach ly: doc RAW read-only, ghi MOI THU duoi BSC_ROOT, dataset/folder moi.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
BSC_ROOT = /content/drive/MyDrive/bsc
Cach ly: doc RAW read-only, ghi MOI THU duoi BSC_ROOT, dataset/folder moi.


## 1. Kiểm splits đã ghim (Phase 0.3 — đã chạy local)

In [3]:
import json
sp = json.load(open(f"{REPO_DIR}/bsc/splits/splits_zib_v1.json"))
print("n_folds:", sp["n_folds"])
print("Ro ri (goc):", sp["leak_audit"]["n_groups_leaked"], "/ 70 dau goi")
spf = json.load(open(f"{REPO_DIR}/bsc/splits/splits_zib_v1_fixed.json"))
print("Ro ri (da sua):", spf["leak_audit"]["n_groups_leaked"], "(phai = 0)")
# Copy len Drive de cac notebook sau dung
import shutil
for f in ["splits_zib_v1.json","splits_zib_v1_fixed.json"]:
    shutil.copy(f"{REPO_DIR}/bsc/splits/{f}", f"{BSC_ROOT}/splits/{f}")
print("Da copy splits len Drive.")

n_folds: 5
Ro ri (goc): 51 / 70 dau goi
Ro ri (da sua): 0 (phai = 0)
Da copy splits len Drive.


## 2. Giải nén baseline từ 15 GB zip → Drive

Zip đang ở local (`nnUNet_results/Dataset020_KneeUnion-*.zip`). **Upload chúng lên
Drive trước** (vd `MyDrive/nnResult_zips/`), rồi chạy cell này để giải nén B0/B1/B2
vào `BSC_ROOT/baselines/`.

B0 = 250ep fold_0 (model đã báo cáo). B1 = 150ep fold_0..4 (đủ 5 fold, 544 pred CV).
B2 = Dataset021 ROI cascade (negative baseline).

In [3]:
from bsc import io_utils
ZIP_DIR = "/content/drive/MyDrive/nnUNet_results"    # <-- noi ban upload zip

n = io_utils.unzip_multipart(f"{ZIP_DIR}/Dataset020_KneeUnion-*.zip",
                             f"{BSC_ROOT}/baselines/ds020")
print(f"Giai nen {n} file d020")
io_utils.unzip_multipart(f"{ZIP_DIR}/Dataset021_CartROI-*.zip",
                         f"{BSC_ROOT}/baselines/ds021")

# Kiem checkpoint B0 (250ep fold_0)
import glob
ck = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_250epochs*/fold_0/checkpoint_best.pth",
               recursive=True)
print("B0 checkpoint:", ck)

Giai nen 715 file d020
B0 checkpoint: ['/content/drive/MyDrive/bsc/baselines/ds020/Dataset020_KneeUnion/nnUNetTrainer_250epochs__nnUNetResEncUNetLPlans__3d_fullres/fold_0/checkpoint_best.pth']


## 3. GATE 0 — Kiểm chứng metric trên prediction B0 đã có

Bản 150ep đã có **544 prediction validation** (đường `.../fold_N/validation/*.nii.gz`).
Đối chiếu GT ↔ pred bằng `bsc.metrics`, so với bảng đã công bố.

**QUAN TRỌNG:** dùng `hd95(mode="pooled")` để khớp `surf()` cũ. Nếu Dice/ASSD femoral
cartilage khớp bảng trong ±0.005 ⇒ **Gate 0 PASS**.

In [4]:
import glob, os
print("RAW hien tai:", RAW, "->", os.path.isdir(RAW))
print("vai file d001:", [os.path.basename(p) for p in glob.glob(f"{RAW}/labelsTr/*.nii.gz")[:3]])

D020 = "/content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion"   # kiem lai duong nay
print("\nd020 ton tai:", os.path.isdir(f"{D020}/labelsTr"))
print("vai file d020:", [os.path.basename(p) for p in glob.glob(f"{D020}/labelsTr/*.nii.gz")[:3]])
print("so oaizib_* trong d020:", len(glob.glob(f"{D020}/labelsTr/oaizib_*.nii.gz")))

import numpy as np
from bsc import io_utils
g, _ = io_utils.load_nii(glob.glob(f"{D020}/labelsTr/oaizib_*.nii.gz")[0])
print("label co trong GT d020:", np.unique(g))   # ky vong: [0 1 2 3 4 5 ...]


RAW hien tai: /content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA -> True
vai file d001: ['oaizib_005.nii.gz', 'oaizib_105.nii.gz', 'oaizib_078.nii.gz']

d020 ton tai: True
vai file d020: ['oaizib_001.nii.gz', 'oaizib_002.nii.gz', 'oaizib_003.nii.gz']
so oaizib_* trong d020: 404
label co trong GT d020: [0 1 2 3 4 5 6 7 8]


In [4]:
import csv, glob, os
import numpy as np
from tqdm import tqdm
from bsc import io_utils, metrics

PRED_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/",
                     recursive=True)[0]

# GT phai lay tu CHINH dataset da sinh ra prediction. Dataset001_KneeOA cung co case
# ten oaizib_* nhung dung scheme 4-lop (1,3 = xuong) - doc no bang bang label union
# se ra so TRONG HOP LY nhung VO NGHIA cho med/lat tibial cart. Xem preflight ben duoi.
NNRAW = os.path.dirname(RAW)
GT_LABELS = f"{NNRAW}/Dataset020_KneeUnion/labelsTr"
CKPT = f"{BSC_ROOT}/runs/gate0_percase.csv"   # Drive-first: song sot khi dut ket noi

CART = {"femoral_cart": 2, "med_tib_cart": 4, "lat_tib_cart": 5}
PUBLISHED = {"femoral_cart": (0.891, 0.21), "med_tib_cart": (0.852, 0.27),
             "lat_tib_cart": (0.868, 0.27)}

preds = sorted(glob.glob(f"{PRED_DIR}/fold_*/validation/oaizib_*.nii.gz"))
print(f"{len(preds)} prediction OAI-ZIB")
print(f"GT: {GT_LABELS}")

# --- Preflight: khop bang label TRUOC khi dot 2 tieng dong ho -----------------
_probe = f"{GT_LABELS}/{os.path.basename(preds[0])}"
assert os.path.exists(_probe), f"Khong thay GT: {_probe}"
_g, _ = io_utils.load_nii(_probe)
_have = set(int(v) for v in np.unique(_g))
_need = set(CART.values())
assert _need <= _have, (
    f"GT thieu label {sorted(_need - _have)} (co: {sorted(_have)}).\n"
    f"{GT_LABELS} khong dung union 8-class."
)
print(f"Preflight OK - label trong GT: {sorted(_have)}")

# --- Resume: bo qua ca da tinh xong -------------------------------------------
os.makedirs(os.path.dirname(CKPT), exist_ok=True)
done, rows = set(), {c: {"dice": [], "assd": []} for c in CART}
if os.path.exists(CKPT):
    with open(CKPT, newline="") as f:
        for r in csv.DictReader(f):
            done.add(r["case"])
            rows[r["cls"]]["dice"].append(float(r["dice"]))
            rows[r["cls"]]["assd"].append(float(r["assd"]))
    print(f"Tiep tuc: da co {len(done)} ca trong {CKPT}")

todo = [p for p in preds if os.path.basename(p)[:-len(".nii.gz")] not in done]
n_skip_nogt, n_skip_empty = 0, 0

with open(CKPT, "a", newline="") as fh:
    w = csv.writer(fh)
    if not done:
        w.writerow(["case", "cls", "dice", "assd"])
    for pf in tqdm(todo, desc="Gate 0"):
        cid = os.path.basename(pf)[:-len(".nii.gz")]
        gtf = f"{GT_LABELS}/{cid}.nii.gz"
        if not os.path.exists(gtf):
            n_skip_nogt += 1
            continue
        gt, sp = io_utils.load_nii(gtf)
        pr, _ = io_utils.load_nii(pf)
        for c, lab in CART.items():
            g, p = (gt == lab), (pr == lab)
            if not g.any():
                n_skip_empty += 1
                continue
            d, a = metrics.dice(g, p), metrics.assd(g, p, sp)
            rows[c]["dice"].append(d)
            rows[c]["assd"].append(a)
            w.writerow([cid, c, d, a])
        fh.flush()   # ghi ngay tung ca: dut ket noi van con du lieu

# --- Khong de rows rong lang le troi thanh nan --------------------------------
if n_skip_nogt:
    print(f"CANH BAO: {n_skip_nogt}/{len(todo)} ca KHONG co GT trong {GT_LABELS}")
assert any(rows[c]["dice"] for c in CART), (
    f"Khong tinh duoc ca nao. Khong co GT: {n_skip_nogt}, lop vang trong GT: {n_skip_empty}. "
    f"Kiem GT_LABELS ({GT_LABELS}) va ten case."
)

print(f"\n{'lop':<16}{'n':>5}{'Dice':>8}{'ASSD':>8}   (cong bo: fem .891/.21, med .852/.27, lat .868/.27)")
gate0_ok = True
for c in CART:
    n = len(rows[c]["dice"])
    if n == 0:
        print(f"{c:<16}{0:>5}{'--':>8}{'--':>8}   KHONG CO DU LIEU - lop vang trong GT?")
        gate0_ok = False
        continue
    d, a = np.mean(rows[c]["dice"]), np.nanmean(rows[c]["assd"])
    pd_, pa = PUBLISHED[c]
    ok = abs(d - pd_) < 0.02   # CV nen sat CV; test set moi la +-0.005
    gate0_ok &= ok
    print(f"{c:<16}{n:>5}{d:>8.3f}{a:>8.3f}   published {pd_:.3f}/{pa:.2f}  {'OK' if ok else 'LECH!'}")

print(f"\nGATE 0 (CV, +-0.02): {'PASS' if gate0_ok else 'FAIL - dieu tra truoc khi tiep'}")
print(f"Per-case da ghi: {CKPT}")
print("Luu y: day la so CV (bi thoi phong boi ro ri iMorph, nhung ZIB single-timepoint")
print("nen ZIB CV van sat). So test that lam o cuoi, doi chieu +-0.005.")

404 prediction OAI-ZIB
GT: /content/drive/MyDrive/nnUNet_raw/Dataset020_KneeUnion/labelsTr
Preflight OK - label trong GT: [0, 1, 2, 3, 4, 5, 6, 7, 8]
Tiep tuc: da co 276 ca trong /content/drive/MyDrive/bsc/runs/gate0_percase.csv


Gate 0:   0%|          | 0/128 [00:00<?, ?it/s]


lop                 n    Dice    ASSD   (cong bo: fem .891/.21, med .852/.27, lat .868/.27)
femoral_cart      404   0.889   0.271   published 0.891/0.21  OK
med_tib_cart      404   0.844   0.280   published 0.852/0.27  OK
lat_tib_cart      404   0.861   0.291   published 0.868/0.27  OK

GATE 0 (CV, +-0.02): PASS
Per-case da ghi: /content/drive/MyDrive/bsc/runs/gate0_percase.csv
Luu y: day la so CV (bi thoi phong boi ro ri iMorph, nhung ZIB single-timepoint
nen ZIB CV van sat). So test that lam o cuoi, doi chieu +-0.005.


## 3b. GATE 0 THAT SU — dac trung hoa do lech METRIC: scipy vs SimpleITK

Bang 3a chi chung minh **Dice dung tam**. Du an do o thang **0.04-0.1mm ASSD**, ma bang cu
(0.21mm) tinh bang `surf()` SimpleITK. Phai dac trung hoa do lech scipy-vs-sitk cho tu te.

**Sua hai hieu nham (theo review):**
- ASSD la **trung binh**, khong phai tong => femoral lech nhieu hon KHONG phai vi no lon, ma
  vi no co **ty le mat cong/xien** cao hon (hai impl dat bien vat ly khac cho) + spacing z bat
  doi xung. Day la GIA THUYET, chua khang dinh.
- Gap tuyet doi 0.06mm **KHONG** phai "san nhieu" cua Gate 1. Prize la hieu TRONG cung scipy
  nen offset hang so **triet tieu**: `ε_metric = Prize_scipy − Prize_sitk = δ_B − δ_M`, chi cai
  nay moi lam prize phu thuoc dinh nghia. Prize 0.04mm van do duoc du gap 0.06mm.

Cell duoi: **δ per-case tren ~40 ca**, ghep voi ASSD scipy da co trong `gate0_percase.csv`, bao
cao phan phoi (median/IQR/SD/%duong/bootstrap CI). Offset ON DINH (vd 0.055,0.061,0.059...) =>
chap nhan, chi bao cao scipy nhat quan. Offset TUY CA that thuong => dieu tra (§6 review).

In [ ]:
import csv, glob, os
import numpy as np
import SimpleITK as sitk
from bsc import io_utils, metrics

# --- ASSD SimpleITK, SAO Y merge_s6_evaluate cell 22 (surf) -------------------
def _assd_sitk(gm, pm, spacing_xyz):
    if gm.sum() == 0 or pm.sum() == 0:
        return np.nan
    gi = sitk.GetImageFromArray(gm.astype(np.uint8)); gi.SetSpacing(spacing_xyz)
    pi = sitk.GetImageFromArray(pm.astype(np.uint8)); pi.SetSpacing(spacing_xyz)
    gdm = sitk.Abs(sitk.SignedMaurerDistanceMap(gi, squaredDistance=False, useImageSpacing=True))
    pdm = sitk.Abs(sitk.SignedMaurerDistanceMap(pi, squaredDistance=False, useImageSpacing=True))
    gs = sitk.GetArrayViewFromImage(sitk.LabelContour(gi)).astype(bool)
    ps = sitk.GetArrayViewFromImage(sitk.LabelContour(pi)).astype(bool)
    d1 = sitk.GetArrayViewFromImage(gdm)[ps]; d2 = sitk.GetArrayViewFromImage(pdm)[gs]
    if len(d1) == 0 or len(d2) == 0:
        return np.nan
    return float(np.concatenate([d1, d2]).mean())

PRED_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/", recursive=True)[0]
GT_LABELS = f"{os.path.dirname(RAW)}/Dataset020_KneeUnion/labelsTr"
SCIPY_CSV = f"{BSC_ROOT}/runs/gate0_percase.csv"          # ASSD scipy da co (404 ca)
SITK_CKPT = f"{BSC_ROOT}/runs/gate0b_sitk_assd.csv"       # ASSD sitk (resume duoc)
CART = {"femoral_cart": 2, "med_tib_cart": 4, "lat_tib_cart": 5}
N_CHECK = 40   # ~40 ca trai deu qua 404 (khong chi fold0). ~15-25 phut CPU, co resume.

# ASSD scipy da tinh o Gate 0 - tai su dung, khoi tinh lai
scipy_assd = {}
with open(SCIPY_CSV, newline="") as f:
    for r in csv.DictReader(f):
        scipy_assd[(r["case"], r["cls"])] = float(r["assd"])

# Lay ~N_CHECK ca trai deu qua danh sach (span nhieu fold), chi lay ca co trong CSV scipy
all_preds = sorted(glob.glob(f"{PRED_DIR}/fold_*/validation/oaizib_*.nii.gz"))
have = [p for p in all_preds if (os.path.basename(p)[:-7], "femoral_cart") in scipy_assd]
stride = max(1, len(have) // N_CHECK)
preds = have[::stride][:N_CHECK]

# Resume phia sitk
sitk_assd, done = {}, set()
if os.path.exists(SITK_CKPT):
    with open(SITK_CKPT, newline="") as f:
        for r in csv.DictReader(f):
            sitk_assd[(r["case"], r["cls"])] = float(r["assd_sitk"])
            done.add(r["case"])
todo = [p for p in preds if os.path.basename(p)[:-7] not in done]
print(f"Dac trung hoa δ tren {len(preds)} ca ({len(done)} da co, {len(todo)} can chay)")

from tqdm import tqdm
with open(SITK_CKPT, "a", newline="") as fh:
    w = csv.writer(fh)
    if not done:
        w.writerow(["case", "cls", "assd_sitk"])
    for pf in tqdm(todo, desc="sitk ASSD"):
        cid = os.path.basename(pf)[:-7]
        gt, sp = io_utils.load_nii(f"{GT_LABELS}/{cid}.nii.gz")
        pr, _ = io_utils.load_nii(pf)
        sp_xyz = (sp[2], sp[1], sp[0])
        for c, lab in CART.items():
            g, p = (gt == lab), (pr == lab)
            if not g.any() or not p.any():
                continue
            a = _assd_sitk(g, p, sp_xyz)
            sitk_assd[(cid, c)] = a
            w.writerow([cid, c, a])
        fh.flush()

# --- Phan phoi δ = scipy - sitk theo tung lop --------------------------------
def _boot_ci(d, n=10000, seed=0):
    b = metrics.paired_bootstrap(np.zeros_like(d), d, n_boot=n, seed=seed)
    return b["ci_low"], b["ci_high"]

print(f"\nδ = ASSD_scipy − ASSD_sitk  (mm), per-case, {len(preds)} ca")
print(f"{'lop':<14}{'n':>4}{'mean':>8}{'median':>8}{'sd':>7}{'IQR':>14}{'%duong':>8}{'boot95%CI':>18}")
for c in CART:
    d = np.array([scipy_assd[(os.path.basename(p)[:-7], c)] - sitk_assd[(os.path.basename(p)[:-7], c)]
                  for p in preds
                  if (os.path.basename(p)[:-7], c) in sitk_assd and np.isfinite(sitk_assd[(os.path.basename(p)[:-7], c)])], float)
    if d.size == 0:
        print(f"{c:<14}  (khong co ca hop le)"); continue
    q1, q3 = np.percentile(d, [25, 75])
    lo, hi = _boot_ci(d)
    print(f"{c:<14}{d.size:>4}{d.mean():>+8.4f}{np.median(d):>+8.4f}{d.std():>7.4f}"
          f"{f'[{q1:+.3f},{q3:+.3f}]':>14}{(d > 0).mean():>7.0%}{f'[{lo:+.3f},{hi:+.3f}]':>18}")

print("\nDoc ket qua (review §4.2):")
print("  SD nho + IQR hep + %duong~100% + median~mean => OFFSET ON DINH. Chap nhan,")
print("     bao cao scipy nhat quan, KHONG dat canh 0.21 sitk. Prize Gate 1 van hop le.")
print("  SD lon / IQR rong / %duong ~50% / co ca doi dau => LECH TUY CA. Xem §6 review:")
print("     phantom huong be mat, resample isotropic, tuong quan δ voi do cong.")

## 4. Điều tra KL grade (Phase 0.6)

User xác nhận KL là label của OAI-ZIB trên HuggingFace `YongchengYAO/OAIZIB-CM`.
Tải `info.zip` và dò cột KL + patient ID để map `oaizib_XXX` → KL.

**Thiết kế không phụ thuộc cứng vào KL** — bin độ dày là stratifier chính. KL là bổ sung
nếu tìm được mapping.

In [5]:
from huggingface_hub import snapshot_download
info_dir = snapshot_download(repo_id="YongchengYAO/OAIZIB-CM", repo_type="dataset",
                             allow_patterns=["*.csv","*.json","*.txt","info*"],
                             local_dir="/content/oaizib_info")
import glob
for f in glob.glob(f"{info_dir}/**/*", recursive=True):
    if os.path.isfile(f) and f.split(".")[-1] in ("csv","json","txt"):
        print(f, os.path.getsize(f), "bytes")
# TODO: mo file metadata, tim cot KL + cot noi voi oaizib_XXX.
# Neu tim thay: ghi BSC_ROOT/splits/kl_map.json = {case_id: kl_grade}
# Neu khong: bo qua - bin do day la stratifier chinh.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

## Gate 0 checklist
- [ ] `splits_zib_v1*.json` trên Drive
- [ ] B0/B1/B2 giải nén vào `BSC_ROOT/baselines/`
- [ ] **Metric femoral_cart khớp bảng đã công bố** ← quan trọng nhất
- [ ] KL map (tùy chọn)

**Không có gì quan trọng nằm ở `/content/`** (ngoài `/content/drive/`). ✅ → sang bsc_01.